In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mplfinance as mpf

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE

In [ ]:
# Load Processed Data
file_path = "C:/Users/Public/stockvolatilitydetector/processed_data.csv"
df = pd.read_csv(file_path)

In [ ]:
# Feature Engineering: RSI, MACD, ATR
df["RSI"] = 100 - (100 / (1 + df["Close"].pct_change().rolling(window=14).mean()))
df["MACD"] = df["Close"].ewm(span=12).mean() - df["Close"].ewm(span=26).mean()
df["ATR"] = (df["High"] - df["Low"]).rolling(window=14).mean()

In [ ]:
# Rolling Std Using NumPy
window_size = 14
rolling_std = np.std(np.lib.stride_tricks.sliding_window_view(df["Close"], window_size), axis=1)
df = df.iloc[window_size - 1:].copy()
df["Rolling_Std_NumPy"] = rolling_std

In [ ]:
# Define Features and Target
X = df[["Close", "Volume", "RSI", "MACD", "ATR"]]
y = (df["Rolling_Std_NumPy"] > df["Rolling_Std_NumPy"].quantile(0.75)).astype(int)

In [ ]:
X = X.dropna().copy()
y = y.loc[X.index]

In [ ]:
# Balance Data with SMOTE
smote = SMOTE(sampling_strategy=0.75, random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

In [ ]:
# Split Dataset
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=42)

In [ ]:
# Train Models
model_rf = RandomForestClassifier(n_estimators=5, max_depth=1, random_state=42)
model_rf.fit(X_train, y_train)

In [ ]:
model_lr = LogisticRegression(penalty='l2', solver='liblinear')
model_lr.fit(X_train, y_train)

In [ ]:
model_svm = SVC(kernel='rbf', C=0.25)
model_svm.fit(X_train, y_train)

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7]
}
grid_search = GridSearchCV(GradientBoostingClassifier(random_state=42), param_grid, cv=5)
grid_search.fit(X_train, y_train)
best_params = grid_search.best_params_
model_gb = GradientBoostingClassifier(**best_params, random_state=42)
model_gb.fit(X_train, y_train)

In [ ]:
# Evaluate with Accuracy
print(f"\n✅ RF Accuracy: {model_rf.score(X_test, y_test):.2f}")
print(f"✅ LR Accuracy: {model_lr.score(X_test, y_test):.2f}")
print(f"✅ SVM Accuracy: {model_svm.score(X_test, y_test):.2f}")
print(f"✅ GB Accuracy: {model_gb.score(X_test, y_test):.2f}")

In [ ]:
# Detailed Evaluation: Confusion Matrix + Report (Random Forest)
y_pred_rf = model_rf.predict(X_test)
print("\n📌 Confusion Matrix (Random Forest):")
print(confusion_matrix(y_test, y_pred_rf))

In [ ]:
print("\n📌 Classification Report (Random Forest):")
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Rolling Std Histogram
plt.hist(df["Rolling_Std_NumPy"], bins=30)
plt.title("Distribution of Rolling Standard Deviation (NumPy)")
plt.show()

In [ ]:
# Candlestick Plot
ohlc_data = df[["Date", "Open", "High", "Low", "Close", "Volume"]].copy()
ohlc_data["Date"] = pd.to_datetime(ohlc_data["Date"])
ohlc_data.set_index("Date", inplace=True)
mpf.plot(
    ohlc_data.tail(50),
    type="candle",
    volume=True,
    style="charles",
    title="Stock Price Movements",
    ylabel="Stock Price",
    ylabel_lower="Volume"
)

In [ ]:
# Feature Correlation
print("\n📊 Feature Correlation Matrix:")
print(X.corr())